# WorldQuant BRAIN Alpha Research Pipeline — Google Colab

這份 Notebook 依照你提供的規格建立完整五檔 infrastructure，並調整為可在 Colab 直接執行。

> **登入／生物辨識說明**：Notebook 不使用瀏覽器自動化、不嘗試繞過 CAPTCHA、生物辨識、MFA 或任何帳號安全機制。它只使用官方 API session authentication；如果你的帳號登入流程要求額外驗證，請先依 WorldQuant BRAIN 官方流程完成驗證，或使用已獲授權的 API/session 方式。

建議先用 `USE_FAKE_WORKER = True` 測完整 pipeline；確認你的 API 權限與登入方式後，再切換成真實 `Worker`。


In [1]:
!pip -q install requests numpy pandas tqdm

In [2]:
import os, sys
BASE_DIR = "/content/brain_infra"
DB_PATH = os.path.join(BASE_DIR, "db")
os.makedirs(DB_PATH, exist_ok=True)
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
print(BASE_DIR)


/content/brain_infra


In [3]:
%%writefile /content/brain_infra/alpha.py
import os, json, shutil
from enum import Enum

DB_PATH = "/content/brain_infra/db"

class AlphaStage(Enum):
    NONE = "non_exist"
    PENDING = os.path.join(DB_PATH, "pending")
    COMPLETE = os.path.join(DB_PATH, "complete")
    ERROR = os.path.join(DB_PATH, "error")

def init_db():
    for stage in (AlphaStage.PENDING, AlphaStage.COMPLETE, AlphaStage.ERROR):
        os.makedirs(stage.value, exist_ok=True)

class Alpha:
    def __init__(self, name: str, payload: dict,
                 alpha_stage: AlphaStage = AlphaStage.PENDING,
                 result=None):
        self._json = {
            "name": name,
            "payload": payload,
            "stage": alpha_stage.value,
            "result": {} if result is None else result,
        }

    @property
    def name(self): return self._json["name"]
    @property
    def payload(self): return self._json["payload"]
    @property
    def stage(self): return AlphaStage(self._json["stage"])
    @property
    def result(self): return self._json["result"]
    @property
    def filename(self): return f"{self.name}.json"
    @property
    def filepath(self): return os.path.join(self.stage.value, self.filename)
    @property
    def _tmp_filepath(self): return os.path.join(self.stage.value, f"tmp_{self.filename}")

    def dump(self):
        os.makedirs(self.stage.value, exist_ok=True)
        with open(self._tmp_filepath, "w", encoding="utf-8") as f:
            json.dump(self._json, f, ensure_ascii=False)
        shutil.move(self._tmp_filepath, self.filepath)

    @classmethod
    def load(cls, filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        return cls(data["name"], data["payload"],
                   AlphaStage(data["stage"]), data.get("result", {}))

    def update_stage(self, new_stage):
        if os.path.exists(self.filepath):
            os.remove(self.filepath)
        self._json["stage"] = new_stage.value
        self.dump()


Overwriting /content/brain_infra/alpha.py


In [4]:
%%writefile /content/brain_infra/scorer.py
from abc import ABC, abstractmethod

class ScorerBase(ABC):
    @abstractmethod
    def get_name(self) -> str:
        pass

    @abstractmethod
    def score(self, alpha) -> float:
        pass

    def score_list(self, alpha_list):
        return [self.score(alpha) for alpha in alpha_list._alphas]

class SharpeScorer(ScorerBase):
    def get_name(self): return "sharpe"
    def score(self, alpha):
        return alpha.result.get("is", {}).get("sharpe", -100)

class FitnessScorer(ScorerBase):
    def get_name(self): return "fitness"
    def score(self, alpha):
        return alpha.result.get("is", {}).get("fitness", -100)


Overwriting /content/brain_infra/scorer.py


In [5]:
%%writefile /content/brain_infra/alpha_list.py
import os, time
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
from alpha import Alpha, AlphaStage

class AlphaList:
    def __init__(self, alphas: List[Alpha], names: List[str]):
        self._alphas = alphas
        self._names = [alpha.filename for alpha in alphas]

    @staticmethod
    def check_alpha_stage(filename: str) -> AlphaStage:
        for stage in (AlphaStage.PENDING, AlphaStage.COMPLETE, AlphaStage.ERROR):
            if os.path.exists(os.path.join(stage.value, filename)):
                return stage
        return AlphaStage.NONE

    def update_and_check_status(self) -> Tuple[int, int, int, int]:
        counts = {s: 0 for s in AlphaStage}
        for i, alpha in enumerate(self._alphas):
            stage = self.check_alpha_stage(alpha.filename)
            counts[stage] += 1
            if stage != alpha.stage and stage != AlphaStage.NONE:
                self._alphas[i] = Alpha.load(os.path.join(stage.value, alpha.filename))
        return (counts[AlphaStage.NONE], counts[AlphaStage.PENDING],
                counts[AlphaStage.COMPLETE], counts[AlphaStage.ERROR])

    def sim_and_wait(self):
        for alpha in self._alphas:
            if self.check_alpha_stage(alpha.filename) == AlphaStage.NONE:
                alpha.dump()

        pbar = tqdm(total=len(self._alphas), desc="Waiting for simulations")
        last_done = 0
        while True:
            _, pending, complete, error = self.update_and_check_status()
            done = complete + error
            pbar.update(max(0, done - last_done))
            last_done = done
            if done == len(self._alphas):
                break
            time.sleep(1)
        pbar.close()

    def get_alphas(self) -> Dict[str, Alpha]:
        self.update_and_check_status()
        return dict(zip(self._names, self._alphas))


Overwriting /content/brain_infra/alpha_list.py


In [6]:
%%writefile /content/brain_infra/worker.py
import os, pickle, datetime, random, time
from datetime import timedelta
from typing import List, Dict, Tuple
import requests
from alpha import Alpha, AlphaStage, DB_PATH

_SESSION_CACHE = os.path.join(DB_PATH, "session_cache.pkl")
TIMEOUT = 4
SIM_LIMIT = 3
MULTI_SIM_SIZE = 10
API_BASE = "https://api.worldquantbrain.com"

class BrainSession:
    def __init__(self, username=None, password=None):
        self.username = username or os.environ.get("WQB_USERNAME")
        self.password = password or os.environ.get("WQB_PASSWORD")
        self.login()

    def _session_is_valid(self) -> bool:
        return (os.path.exists(_SESSION_CACHE) and
                datetime.datetime.now() -
                datetime.datetime.fromtimestamp(os.path.getmtime(_SESSION_CACHE))
                < timedelta(hours=TIMEOUT))

    def login(self, force_relogin=False):
        if not force_relogin and self._session_is_valid():
            with open(_SESSION_CACHE, "rb") as f:
                self._sess = pickle.load(f)
            return

        self._sess = requests.Session()

        # 官方 API authentication；不使用 Selenium、不繞過 CAPTCHA/MFA/生物辨識。
        if self.username and self.password:
            resp = self._sess.post(
                f"{API_BASE}/authentication",
                auth=(self.username, self.password),
                timeout=30
            )
            if resp.status_code not in (200, 201):
                raise Exception(
                    f"Authentication failed: {resp.status_code}. "
                    "Complete any required account verification through the official flow."
                )
        else:
            raise RuntimeError(
                "請設定 WQB_USERNAME 與 WQB_PASSWORD。"
                "此 Notebook 不會自動處理或繞過 MFA/CAPTCHA/生物辨識。"
            )

        with open(_SESSION_CACHE, "wb") as f:
            pickle.dump(self._sess, f)

    def _request(self, method, url, **kwargs):
        resp = getattr(self._sess, method)(url, **kwargs)
        if resp.status_code == 401:
            self.login(force_relogin=True)
            resp = getattr(self._sess, method)(url, **kwargs)
        if resp.status_code in (200, 201):
            return resp
        raise Exception(f"HTTP {resp.status_code}: {dict(resp.headers)}")

    def get(self, url, **kwargs): return self._request("get", url, **kwargs)
    def post(self, url, **kwargs): return self._request("post", url, **kwargs)
    def patch(self, url, **kwargs): return self._request("patch", url, **kwargs)
    def delete(self, url, **kwargs): return self._request("delete", url, **kwargs)

class Worker:
    def __init__(self, multi_simulation: bool = True):
        self.sess = BrainSession()
        self.multi_simulation = multi_simulation

    @staticmethod
    def get_pending_filepaths(running_filepaths: List[str]) -> List[str]:
        if not os.path.exists(AlphaStage.PENDING.value):
            return []
        ans = []
        for fn in os.listdir(AlphaStage.PENDING.value):
            fp = os.path.join(AlphaStage.PENDING.value, fn)
            if fn.endswith(".json") and not fn.startswith("tmp") and fp not in running_filepaths:
                ans.append(fp)
        return sorted(ans)

    def _post_payload(self, alpha):
        try:
            r = self.sess.post(f"{API_BASE}/simulations", json=alpha.payload)
            return (True, r.headers.get("Location", "")) if r.headers.get("Location") else (False, "")
        except Exception:
            return (False, "")

    def _post_multi_payload(self, alphas):
        if not 2 <= len(alphas) <= MULTI_SIM_SIZE:
            raise ValueError("multi-simulation batch must contain 2..10 alphas")
        try:
            r = self.sess.post(f"{API_BASE}/simulations", json=[a.payload for a in alphas])
            return (True, r.headers.get("Location", "")) if r.headers.get("Location") else (False, "")
        except Exception:
            return (False, "")

    def _get_status(self, location_url):
        try:
            r = self.sess.get(location_url)
            data = r.json()
            retry = r.headers.get("Retry-After", "0")
            if retry == "0":
                return ((True, "0", data["alpha"]) if "alpha" in data
                        else (False, "0", ""))
            return True, retry, ""
        except Exception:
            return False, "0", ""

    def _get_multi_status(self, location_url):
        try:
            r = self.sess.get(location_url)
            data = r.json()
            retry = r.headers.get("Retry-After", "0")
            if retry == "0":
                return ((True, "0", data["children"]) if "children" in data
                        else (False, "0", []))
            return True, retry, []
        except Exception:
            return False, "0", []

    def _get_result(self, alpha_id):
        return self.sess.get(f"{API_BASE}/alphas/{alpha_id}").json()

    def _mark_error(self, filepath):
        a = Alpha.load(filepath)
        a.update_stage(AlphaStage.ERROR)

    def _complete(self, filepath, alpha_id):
        a = Alpha.load(filepath)
        a._json["result"] = self._get_result(alpha_id)
        a.update_stage(AlphaStage.COMPLETE)

    def run(self):
        running_simulations: Dict[str, List[str]] = {}
        try:
            while True:
                while len(running_simulations) < SIM_LIMIT:
                    running = [fp for fps in running_simulations.values() for fp in fps]
                    pending = self.get_pending_filepaths(running)
                    if not pending:
                        break

                    if not self.multi_simulation:
                        batch = pending[:1]
                        alphas = [Alpha.load(batch[0])]
                        ok, loc = self._post_payload(alphas[0])
                    else:
                        batch = pending[:MULTI_SIM_SIZE]
                        alphas = [Alpha.load(fp) for fp in batch]
                        if len(batch) == 1:
                            ok, loc = self._post_payload(alphas[0])
                        else:
                            ok, loc = self._post_multi_payload(alphas)

                    if ok:
                        running_simulations[loc] = batch
                    else:
                        for fp in batch:
                            self._mark_error(fp)
                    time.sleep(1)

                if not running_simulations:
                    if not self.get_pending_filepaths([]):
                        break
                    time.sleep(5)
                    continue

                for loc, filepaths in list(running_simulations.items()):
                    if len(filepaths) == 1:
                        ok, retry, alpha_id = self._get_status(loc)
                        if not ok:
                            self._mark_error(filepaths[0])
                            del running_simulations[loc]
                            break
                        if retry == "0":
                            self._complete(filepaths[0], alpha_id)
                            del running_simulations[loc]
                            break
                        time.sleep(float(retry))
                    else:
                        ok, retry, children = self._get_multi_status(loc)
                        if not ok:
                            for fp in filepaths:
                                self._mark_error(fp)
                            del running_simulations[loc]
                            break
                        if retry == "0":
                            for child_id, fp in zip(children, filepaths):
                                ok2, retry2, alpha_id = self._get_status(
                                    f"{API_BASE}/simulations/{child_id}"
                                )
                                if not ok2 or retry2 != "0":
                                    self._mark_error(fp)
                                else:
                                    self._complete(fp, alpha_id)
                            del running_simulations[loc]
                            break
                        time.sleep(float(retry))
        finally:
            for loc in list(running_simulations):
                try:
                    self.sess.delete(loc)
                except Exception:
                    pass

class FakeWorker(Worker):
    def __init__(self, multi_simulation: bool = True):
        self.sess = requests.Session()
        self.multi_simulation = multi_simulation

    def _post_payload(self, alpha): return True, "https://fake.url"
    def _post_multi_payload(self, alphas): return True, "https://fake.url/multi"
    def _get_status(self, location_url): return True, "0", "fake_id"
    def _get_multi_status(self, location_url):
        return True, "0", ["fake_id"] * MULTI_SIM_SIZE

    def _get_result(self, alpha_id):
        sharpe = random.uniform(-0.6, 2.4)
        return {"is": {
            "sharpe": sharpe, "turnover": random.uniform(0, 1),
            "fitness": sharpe, "returns": 0, "drawdown": 0,
            "margin": 0
        }, "startDate": "2012-07-15"}

    def run(self):
        running_simulations = {}
        try:
            while True:
                pending = self.get_pending_filepaths([])
                if not pending:
                    break
                batch = pending[:MULTI_SIM_SIZE] if self.multi_simulation else pending[:1]
                for fp in batch:
                    a = Alpha.load(fp)
                    a._json["result"] = self._get_result("fake_id")
                    a.update_stage(AlphaStage.COMPLETE)
        finally:
            pass


Overwriting /content/brain_infra/worker.py


In [7]:
%%writefile /content/brain_infra/research_process.py
import random
from abc import ABC, abstractmethod
from typing import List, Dict
import numpy as np
from alpha import Alpha
from alpha_list import AlphaList
from worker import Worker, FakeWorker

class ResearchProcessBase(ABC):
    def __init__(self, process_name, scorer, additional_scorers=None,
                 worker_cls=Worker):
        self.process_name = process_name
        self.scorer = scorer
        self.additional_scorers = additional_scorers or []
        self.worker_cls = worker_cls
        self._alpha_lists = []
        self._score_lists = []

    def generate_alpha_name(self, gen, i):
        return f"{self.process_name}_{gen}_{i}"

    def sim_alphas(self, alphas):
        alpha_list = AlphaList(alphas, [a.name for a in alphas])
        self._alpha_lists.append(alpha_list)

        for a in alphas:
            if AlphaList.check_alpha_stage(a.filename).name == "NONE":
                a.dump()

        self.worker_cls().run()
        alpha_list.sim_and_wait()

        alpha_dict = alpha_list.get_alphas()
        scores = {name: self.scorer.score(alpha)
                  for name, alpha in alpha_dict.items()}
        self._score_lists.append(scores)
        return scores

    @abstractmethod
    def run(self):
        pass

class GeneticAlgorithmProcess(ResearchProcessBase):
    def __init__(self, name, scorer, alpha_template, alpha_space,
                 alpha_settings,
                 ga_config=None, worker_cls=Worker):
        super().__init__(name, scorer, worker_cls=worker_cls)
        self.template = alpha_template
        self.alpha_space = alpha_space
        self.alpha_settings = alpha_settings
        self.ga_config = ga_config or {
            "generation": 15, "population": 50,
            "select_rate": 0.5, "mutation_prob": 0.05
        }
        self._generation_genes = []

    def _generate_expr(self, gene):
        expr = self.template
        for key, value in gene.items():
            expr = expr.replace(key, value)
        return expr

    def _generation_iter(self, gen_i, genes):
        pop = self.ga_config["population"]
        if not genes:
            genes = [{k: random.choice(v) for k, v in self.alpha_space.items()}
                     for _ in range(pop)]

        gene_map = {}
        alphas = []
        for i, gene in enumerate(genes):
            name = self.generate_alpha_name(gen_i, i)
            gene_map[name] = gene.copy()
            expr = self._generate_expr(gene)
            payload = dict(self.alpha_settings)
            payload["regular"] = expr
            alphas.append(Alpha(name, payload))

        self._generation_genes.append(gene_map)
        return self.sim_alphas(alphas)

    def _init_population(self):
        return self._generation_iter(0, [])

    def _select(self, scores):
        threshold = np.quantile(list(scores.values()),
                                self.ga_config["select_rate"])
        return [name for name, score in scores.items() if score >= threshold]

    def _crossover(self, parent_genes):
        return {k: random.choice(parent_genes)[k]
                for k in self.alpha_space}

    def _mutate(self, gene):
        out = gene.copy()
        for k, values in self.alpha_space.items():
            if random.random() < self.ga_config["mutation_prob"]:
                out[k] = random.choice(values)
        return out

    def run(self):
        scores = self._init_population()
        for gen_i in range(1, self.ga_config["generation"]):
            survivors = self._select(scores)
            prev_genes = self._generation_genes[-1]
            parents = [prev_genes[name[:-5] if name.endswith(".json") else name]
                       for name in survivors]
            if not parents:
                parents = list(prev_genes.values())

            genes = []
            for _ in range(self.ga_config["population"]):
                genes.append(self._mutate(self._crossover(parents)))
            scores = self._generation_iter(gen_i, genes)
        return self.get_all_results()

    def get_all_results(self):
        rows = []
        for alpha_list in self._alpha_lists:
            for filename, alpha in alpha_list.get_alphas().items():
                rows.append({
                    "filename": filename,
                    "name": alpha.name,
                    "stage": alpha.stage.name,
                    "expression": alpha.payload.get("regular", ""),
                    "sharpe": alpha.result.get("is", {}).get("sharpe", -100),
                    "fitness": alpha.result.get("is", {}).get("fitness", -100),
                    "turnover": alpha.result.get("is", {}).get("turnover", None),
                })
        return rows


Overwriting /content/brain_infra/research_process.py


In [ ]:
import os
import getpass

os.environ["WQB_USERNAME"] = input("WorldQuant username: ")
os.environ["WQB_PASSWORD"] = getpass.getpass("WorldQuant password: ")

print("帳號已設定，密碼不會顯示。")

In [9]:
import os, pickle, requests
from urllib.parse import urljoin
import worker

worker.INTERACTIVE = True          # 背景執行緒要跑時改成 False

def _is_logged_in(resp):
    """成功的唯一訊號：201 且 body 帶 user 物件。
    注意 POST /authentication 不論登入與否都回 401 + 新的 inquiry，
    所以它不能拿來當「檢查登入狀態」用。"""
    if resp.status_code not in (200, 201):
        return False
    try:
        body = resp.json()
    except ValueError:
        return False
    return isinstance(body, dict) and "user" in body

def _login_with_persona(self, force_relogin=False):
    if not force_relogin and self._session_is_valid():
        with open(worker._SESSION_CACHE, "rb") as f:
            self._sess = pickle.load(f)
        return

    if not (self.username and self.password):
        raise RuntimeError("請先設定 WQB_USERNAME 與 WQB_PASSWORD")

    self._sess = requests.Session()
    self._sess.auth = (self.username, self.password)
    auth_url = f"{worker.API_BASE}/authentication"
    persona_url = f"{auth_url}/persona"

    resp = self._sess.post(auth_url, timeout=30)
    if not _is_logged_in(resp):
        try:
            body = resp.json()
        except ValueError:
            body = {}

        is_persona = (resp.status_code == 401 and
                      (resp.headers.get("WWW-Authenticate") == "persona"
                       or "inquiry" in body))
        if not is_persona:
            raise Exception(f"登入失敗 {resp.status_code}: {resp.text[:300]}")

        location = resp.headers.get("Location")
        browser_url = (urljoin(resp.url, location) if location
                       else f"{persona_url}?inquiry={body.get('inquiry','')}")

        if not worker.INTERACTIVE:
            raise Exception(f"需要生物辨識驗證，請在主執行緒重新登入：{browser_url}")

        print("=" * 72)
        print("BRAIN 需要生物辨識驗證（這不是密碼錯誤，帳密已通過）。")
        print("開啟以下網址完成驗證，然後回來按 Enter：")
        print(f"\n  {browser_url}\n")
        print("=" * 72)
        input("完成後按 Enter... ")

        # 帶著使用者剛完成的那個 inquiry 去換 token。
        # 這一步回 201 就是登入成功，沒有第四步。
        resp = self._sess.post(persona_url, json=body, timeout=30)
        if not _is_logged_in(resp):
            resp = self._sess.post(browser_url, timeout=30)
        if not _is_logged_in(resp):
            raise Exception(f"驗證未被接受 {resp.status_code}: {resp.text[:300]}")

    info = resp.json()
    perms = info.get("permissions", [])
    print(f"✅ 登入成功：{info.get('user', {}).get('id', '?')}")
    if perms:
        print(f"   權限：{', '.join(perms)}")
    if "MULTI_SIMULATION" not in perms:
        print("   ⚠️ 沒有 MULTI_SIMULATION 權限，multi_simulation 要設成 False")

    os.makedirs(os.path.dirname(worker._SESSION_CACHE), exist_ok=True)
    with open(worker._SESSION_CACHE, "wb") as f:
        pickle.dump(self._sess, f)

worker.BrainSession.login = _login_with_persona
worker.TIMEOUT = 3.5               # token 實際是 14400 秒，提早換避免跑到一半失效
print("已套用 persona 補丁")

已套用 persona 補丁


In [ ]:
import os, getpass
os.environ['WQB_USERNAME'] = input('BRAIN email: ')
os.environ['WQB_PASSWORD'] = getpass.getpass('password: ')

In [11]:
from worker import BrainSession

try:
    session = BrainSession()
    print("✅ WorldQuant BRAIN authentication 成功！")
except Exception as e:
    print("❌ Authentication 失敗")
    print(type(e).__name__)
    print(e)

✅ WorldQuant BRAIN authentication 成功！


In [12]:
from alpha import init_db
init_db()

# 清除舊測試結果
import shutil, os
for d in ["pending", "complete", "error"]:
    path = f"/content/brain_infra/db/{d}"
    shutil.rmtree(path, ignore_errors=True)
init_db()
print("Database initialized.")


Database initialized.


In [13]:
template = """
data1 = ts_backfill(<funds_data>, <backfill_days>);
data2 = ts_backfill(<debt_data>, <backfill_days>);
diff = <diff_op>(data1, data2);
alpha = <ts_neut_op>(diff, <ts_neut_days>);
alpha_gp = <group_neut_op>(<group_neut_op>(alpha, <gp1>), <gp2>);
<ts_decay_op>(alpha_gp, <ts_decay_days>)
"""

space = {
    "<funds_data>": ["fnd6_fopo", "fnd6_fopox", "anl4_ffo_flag"],
    "<debt_data>": ["debt_lt", "debt_st", "debt", "anl4_netdebt_mean"],
    "<backfill_days>": ["5", "10", "21", "63", "126"],
    "<group_neut_op>": ["group_zscore", "group_rank", "group_neutralize"],
    "<ts_decay_op>": ["ts_mean", "ts_decay_linear"],
    "<ts_neut_op>": ["ts_rank", "ts_zscore", "ts_av_diff", "ts_delta"],
    "<ts_neut_days>": ["5", "10", "21", "63", "126", "252", "512"],
    "<ts_decay_days>": ["1", "5", "10", "21", "42", "63"],
    "<diff_op>": ["subtract", "divide"],
    "<gp1>": ["market", "sector", "industry", "subindustry"],
    "<gp2>": ["market", "sector", "industry", "subindustry"],
}

alpha_settings = {
    "type": "REGULAR",
    "settings": {
        "instrumentType": "EQUITY",
        "region": "USA",
        "universe": "TOP3000",
        "delay": 1,
        "decay": 0,
        "neutralization": "MARKET",
        "truncation": 0.08,
        "pasteurization": "ON",
        "unitHandling": "VERIFY",
        "nanHandling": "OFF",
        "language": "FASTEXPR",
        "visualization": False
    }
}


In [ ]:
# ===== 先用 FakeWorker 驗證整條 pipeline =====
USE_FAKE_WORKER = False

from scorer import SharpeScorer
from research_process import GeneticAlgorithmProcess
from worker import Worker, FakeWorker

# Fake mode 建議先縮小，避免一次產生大量測試檔案
ga_config = {
    "generation": 2,
    "population": 10,
    "select_rate": 0.5,
    "mutation_prob": 0.05
}

process = GeneticAlgorithmProcess(
    name="usa_d1",
    scorer=SharpeScorer(),
    alpha_template=template,
    alpha_space=space,
    alpha_settings=alpha_settings,
    ga_config=ga_config,
    worker_cls=FakeWorker if USE_FAKE_WORKER else Worker
)

rows = process.run()
len(rows)


In [ ]:
import pandas as pd, os

df = pd.DataFrame(rows).sort_values("sharpe", ascending=False)
output_path = "/content/brain_infra/alphas_sorted_by_sharpe.csv"
df.to_csv(output_path, index=False)

display(df.head(20))
print("CSV saved:", output_path)


In [ ]:
# 下載 CSV
from google.colab import files
files.download("/content/brain_infra/alphas_sorted_by_sharpe.csv")


In [ ]:
# ===== 切換到真實 WorldQuant BRAIN API 前才執行 =====
# 不要把密碼直接寫進 Notebook；執行後輸入即可。
#
# import os, getpass
# os.environ["WQB_USERNAME"] = input("WorldQuant username: ")
# os.environ["WQB_PASSWORD"] = getpass.getpass("WorldQuant password: ")
#
# USE_FAKE_WORKER = False
#
# 注意：
# 1. Notebook 只使用 requests 呼叫官方 authentication API。
# 2. 不使用 Selenium，也不會處理或繞過生物辨識、MFA、CAPTCHA。
# 3. 如果帳號要求額外安全驗證，請依官方流程完成後再取得可用 API session/權限。
# 4. multi-simulation 需要你的帳號具有相應權限。
